In [1]:
import pandas as pd
import numpy as np
import pandas as pd 
import numpy as np
import seaborn as sns 
import pymysql
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.metrics import silhouette_score

# Set seed for reproducibility
np.random.seed(42)

# Generate 200 rows of data
n_rows = 200

data = {
    'Customer_ID': range(1001, 1001 + n_rows),
    'Age': np.random.randint(18, 70, n_rows),
    'Annual_Income_k$': np.random.randint(20, 120, n_rows),
    'Spending_Score': np.random.randint(1, 100, n_rows),
    'Membership_Years': np.random.randint(1, 15, n_rows),
    'Purchase_Frequency': np.random.randint(1, 50, n_rows),
    'Last_Purchase_Days': np.random.randint(1, 365, n_rows),
    'Region': np.random.choice(['North', 'South', 'East', 'West'], n_rows),
    'Preferred_Device': np.random.choice(['Mobile', 'Desktop', 'Tablet'], n_rows),
    'Payment_Method': np.random.choice(['Credit Card', 'PayPal', 'Cash'], n_rows)
}

df = pd.DataFrame(data)

# Save to CSV
df.to_csv('test_business_data.csv', index=False)
print("File 'test_business_data.csv' has been created with 200 rows!")

File 'test_business_data.csv' has been created with 200 rows!


In [2]:
df

,Customer_ID,Age,Annual_Income_k$,Spending_Score,Membership_Years,Purchase_Frequency,Last_Purchase_Days,Region,Preferred_Device,Payment_Method
0,1001,56,89,47,7,6,273,North,Mobile,PayPal
1,1002,69,91,86,14,18,231,South,Mobile,PayPal
2,1003,46,46,23,5,5,86,East,Tablet,Credit Card
3,1004,32,28,66,12,47,126,East,Tablet,PayPal
4,1005,60,81,27,1,25,300,East,Tablet,Credit Card
...,...,...,...,...,...,...,...,...,...,...
195,1196,69,77,99,13,25,261,West,Tablet,PayPal
196,1197,30,86,36,13,37,157,West,Mobile,Cash
197,1198,58,65,82,14,28,47,South,Mobile,Credit Card
198,1199,20,43,96,3,10,324,East,Tablet,PayPal


In [3]:
df.shape

(200, 10)

In [4]:
X = df.drop(columns = ['Customer_ID'])

In [5]:
numeric = ['Age','Annual_Income_k$','Spending_Score','Membership_Years','Purchase_Frequency', 'Last_Purchase_Days']
cat = ['Region', 'Preferred_Device','Payment_Method']

In [6]:
# Preprocessing
trans = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat) # Change is here
])

test_data = trans.fit_transform(X)

In [7]:
test_data.round(3)

array([[ 0.844,  0.656, -0.099, ...,  0.   ,  0.   ,  1.   ],
       [ 1.716,  0.723,  1.23 , ...,  0.   ,  0.   ,  1.   ],
       [ 0.173, -0.796, -0.917, ...,  0.   ,  1.   ,  0.   ],
       ...,
       [ 0.978, -0.155,  1.094, ...,  0.   ,  1.   ,  0.   ],
       [-1.572, -0.897,  1.571, ...,  0.   ,  0.   ,  1.   ],
       [ 0.844, -0.627, -0.883, ...,  0.   ,  0.   ,  1.   ]],
      shape=(200, 16))

In [8]:
cluster_range = range(2, 7)
inertia_list = []
sil_list = []

for k in cluster_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    model = kmeans.fit_predict(test_data)

    # Calculate metrics
    inertia = kmeans.inertia_
    sil_score = silhouette_score(test_data, model)
    
    inertia_list.append(inertia)
    sil_list.append(sil_score)
    
    print(f"Clusters={k} → Inertia={inertia:.2f}, Silhouette Score={sil_score:.3f}")

Clusters=2 → Inertia=1431.42, Silhouette Score=0.107
Clusters=3 → Inertia=1311.18, Silhouette Score=0.100
Clusters=4 → Inertia=1214.07, Silhouette Score=0.107
Clusters=5 → Inertia=1142.78, Silhouette Score=0.110
Clusters=6 → Inertia=1081.43, Silhouette Score=0.108


In [9]:
pipe = Pipeline([
    ('trans', trans),
    ('model',  KMeans(n_clusters=2, random_state=42))
])


In [10]:
pipe.fit(X)

,steps,"[('trans', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
X['cluster'] = pipe.predict(X)

In [12]:
X['cluster']

0      1
1      1
2      0
3      0
4      0
      ..
195    1
196    0
197    1
198    1
199    0
Name: cluster, Length: 200, dtype: int32

In [13]:
Xt = pipe.named_steps['trans'].transform(X)
sil_final = silhouette_score(Xt, X['cluster'])
ch_score = calinski_harabasz_score(Xt, X['cluster'])


# ... previous code ...

# Corrected final section

print(f"Final Silhouette: {sil_final:.3f}")
print(f"Calinski-Harabasz: {ch_score:.3f}") # Fixed the 'prin' and the misplaced text

Final Silhouette: 0.107
Calinski-Harabasz: 25.559
